# Lecture 1: The Storage and Serialization Bridge

**Goal:** Understand why Big Data storage patterns (HDFS, Parquet) don't directly translate to AI training, and learn the serialization techniques that bridge this gap.

---

## 1. The Small Files Problem

### Why GPUs Starve on Small Files
When a deep learning trainer requests individual images over a network (or even from a local SSD), each file access incurs a **seek penalty** — the time to locate and open the file before any data bytes flow. This includes:
- DNS resolution (if remote)
- TCP handshake (SYN → SYN-ACK → ACK)
- TLS negotiation (if HTTPS)
- S3 API authentication (AWS Signature V4)
- Disk head seek or SSD page lookup

Let $N$ = total files, $S_{file}$ = average file size, $T_{seek}$ = per-file latency, $B_{network}$ = bandwidth:

$$T_{total\_independent} = N \times \left(T_{seek} + \frac{S_{file}}{B_{network}}\right)$$

**Example with CIFAR-10:**
- $N = 60{,}000$, $T_{seek} = 50\text{ms}$, $S_{file} = 3\text{KB}$, $B_{network} = 1\text{Gbps}$
- Seek penalty alone: $60{,}000 \times 0.05\text{s} = 3{,}000\text{s}$ (50 minutes of GPU idle!)

### The Solution: Contiguous Sharding
By packing thousands of images into sequential `.tar` archives, we pay the seek penalty only once per shard:

$$T_{total\_sharded} \approx T_{seek\_archive} + \frac{N \times S_{file}}{B_{network}}$$

This is the same principle behind Spark's Parquet files and Hadoop's SequenceFiles — amortize metadata overhead across many records.

### Interactive I/O Latency Simulator
Use the sliders below to explore how different parameters affect total ingestion time. Pay special attention to how the red "Seek Time Penalty" bar dominates for independent files.

In [ ]:
# ============================================================================
# INTERACTIVE I/O LATENCY SIMULATOR
# ============================================================================
# This cell creates an interactive visualization comparing the total time
# to read a dataset stored as independent files vs. contiguous shards.

import numpy as np                      # Numerical computing (arrays, math)
import matplotlib.pyplot as plt          # Plotting library
import ipywidgets as widgets             # Interactive UI widgets for Jupyter
from IPython.display import display      # Renders widgets in the notebook

def plot_io_latency(N=60000, T_seek_ms=50.0, S_file_kb=150.0, B_network_mbps=1000.0):
    """
    Calculates and plots the I/O time for independent files vs. sharded archives.
    
    Parameters:
        N (int): Total number of files/samples in the dataset
        T_seek_ms (float): Time-to-first-byte per file access (milliseconds)
                           Typical values: SSD ~0.1ms, HDD ~10ms, HTTP ~50ms, S3 ~100ms
        S_file_kb (float): Average file size in kilobytes
                           CIFAR-10 images are ~3KB, ImageNet ~150KB
        B_network_mbps (float): Network bandwidth in megabits per second
                                Typical: 1Gbps LAN, 10Gbps datacenter, 25Gbps NVLink
    """
    # Unit conversions to make all calculations in seconds and megabytes
    T_seek = T_seek_ms / 1000.0          # Convert ms → seconds
    S_file = S_file_kb / 1024.0          # Convert KB → MB
    B_nw = B_network_mbps / 8.0          # Convert Mbps → MB/s (8 bits per byte)

    # === BASELINE: Independent Files ===
    # Each file requires its own seek + transfer
    time_seek_baseline = N * T_seek           # Total seek time across all files
    time_transfer = N * (S_file / B_nw)       # Total transfer time (bandwidth-limited)
    total_baseline = time_seek_baseline + time_transfer

    # === AMORTIZED: Contiguous Tar Shards ===
    # We pack 1000 items per shard, so only N/1000 seeks are needed
    shards = max(1, N // 1000)                # Number of .tar shards
    time_seek_amortized = shards * T_seek     # Seek penalty per shard (not per file!)
    total_amortized = time_seek_amortized + time_transfer  # Transfer time is identical

    # --- Plotting ---
    fig, ax = plt.subplots(figsize=(10, 5))
    categories = ['Independent Files', 'Contiguous Shards']
    seek_times = [time_seek_baseline, time_seek_amortized]
    transfer_times = [time_transfer, time_transfer]

    # Stacked horizontal bar chart: transfer time (blue) + seek time (red)
    ax.barh(categories, transfer_times, label='Transfer Time (Bandwidth)', color='#3498db')
    ax.barh(categories, seek_times, left=transfer_times, label='Seek Time Penalty', color='#e74c3c')
    
    ax.set_xlabel('Total Time (Seconds)')
    speedup = total_baseline / total_amortized if total_amortized > 0 else float('inf')
    ax.set_title(f'I/O Bottleneck Simulator | Sharding is {speedup:.1f}x Faster')
    ax.legend()
    plt.tight_layout()
    plt.show()

# Create interactive sliders bound to the plotting function
# widgets.interactive() automatically re-calls the function when any slider moves
latency_ui = widgets.interactive(
    plot_io_latency,
    N=widgets.IntSlider(min=1000, max=1000000, step=1000, value=60000,
                        description='Total Files:'),
    T_seek_ms=widgets.FloatSlider(min=1.0, max=200.0, step=1.0, value=50.0,
                                   description='Seek (ms):'),
    S_file_kb=widgets.FloatSlider(min=10.0, max=5000.0, step=10.0, value=150.0,
                                   description='Size (KB):'),
    B_network_mbps=widgets.FloatSlider(min=100.0, max=10000.0, step=100.0, value=1000.0,
                                        description='BW (Mbps):')
)
display(latency_ui)

---

## 2. Interactive S3 Exploration

### What is boto3?
`boto3` is the official AWS SDK for Python. It provides a Pythonic interface to all AWS services, including S3 (Simple Storage Service). Because MinIO is S3-compatible, we can use the exact same API calls — the only difference is the `endpoint_url` parameter pointing to our local server instead of `s3.amazonaws.com`.

### Key S3 Concepts:
- **Bucket**: A top-level container for objects (like a database schema)
- **Object**: A file stored in a bucket, identified by its key (like a file path)
- **Pagination**: S3 limits list responses to 1,000 objects per page. The paginator automatically handles multi-page responses.
- **IAM Credentials**: `aws_access_key_id` and `aws_secret_access_key` authenticate the client. In production, you'd use IAM roles instead of hardcoded keys.

In [ ]:
# ============================================================================
# S3 BUCKET EXPLORATION VIA BOTO3
# ============================================================================
# This cell connects to our local MinIO instance and lists the uploaded shards.

import boto3  # AWS SDK for Python — works identically with MinIO

# Create an S3 client pointing at our local MinIO server.
# In production, you would omit endpoint_url to use real AWS S3.
# The credentials here match what we set in 00_start_minio.py.
s3 = boto3.client(
    's3',
    endpoint_url='http://127.0.0.1:9000',       # Local MinIO (not AWS)
    aws_access_key_id='minioadmin',              # Username
    aws_secret_access_key='minioadmin'            # Password
)

# S3 list_objects_v2 returns at most 1000 objects per call.
# The paginator automatically issues multiple requests if needed.
# This is a common pattern in AWS APIs to handle large result sets.
paginator = s3.get_paginator('list_objects_v2')
try:
    pages = paginator.paginate(Bucket='cifar-streaming')
    print("Files in 'cifar-streaming':")
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents'][:5]:  # Show first 5 objects
                size_mb = obj['Size'] / 1024 / 1024
                print(f"  - {obj['Key']} ({size_mb:.2f} MB)")
            print("  ... (truncated)")
except Exception as e:
    print(f"Error accessing bucket: {e}")

---

## 3. Streaming Data Pipeline with WebDataset

### How WebDataset Streaming Works
Traditional PyTorch datasets (`torchvision.datasets.CIFAR10`) load data from local disk using random access (seek to file position → read bytes). WebDataset replaces this with **sequential HTTP streaming**:

1. The WebDataset URL pattern `{000000..000049}` expands to 50 shard URLs
2. For each shard, WebDataset opens an HTTP connection and streams the `.tar` bytes
3. As bytes arrive, the tar header is parsed to extract individual samples
4. `.decode("rgb")` converts PNG bytes → numpy float32 array (H×W×3, values 0-1)
5. `.to_tuple("png", "cls")` packages the image and label into a tuple

**Why "shardshuffle=False" here?**
In Lecture 1, we want deterministic, ordered access to verify data integrity. In Lecture 2, we'll enable shard shuffling for training randomness.

### Interactive Stream Explorer
The widget below lets you page through images fetched from the MinIO stream. Notice that no data was downloaded to your local filesystem — every image was decoded on-the-fly from the network stream!

In [ ]:
# ============================================================================
# WEBDATASET STREAMING PIPELINE + INTERACTIVE IMAGE EXPLORER
# ============================================================================
# This cell demonstrates streaming data directly from object storage into
# Python memory without intermediate disk storage.

import webdataset as wds             # Streaming dataset library
import matplotlib.pyplot as plt      # Image visualization
import ipywidgets as widgets         # Interactive slider widget
from IPython.display import display  # Widget rendering

# URL pattern with brace expansion: {000000..000049} generates 50 shard URLs.
# WebDataset will iterate through each shard sequentially, streaming bytes
# over HTTP from our local MinIO instance.
url = "http://127.0.0.1:9000/cifar-streaming/cifar-train-{000000..000049}.tar"

# Build the WebDataset pipeline (lazy — nothing is fetched until iteration)
# Each method adds a processing stage to the pipeline:
dataset = (
    wds.WebDataset(url, shardshuffle=False)  # No shard shuffling (deterministic order)
    .decode("rgb")       # Decode PNG bytes → numpy float32 array, shape (H, W, 3)
                         # Values are normalized to [0, 1] range automatically
    .to_tuple("png", "cls")  # Extract (image_array, class_label) tuples
)

# Pre-fetch 100 images into a buffer for the interactive widget.
# In a real training loop, you'd iterate lazily without buffering.
buffer_images, buffer_labels = [], []

try:
    for idx, (img, label) in enumerate(dataset):
        buffer_images.append(img)
        buffer_labels.append(label)
        if idx == 99:  # Buffer first 100 images
            break

    # CIFAR-10 class names for human-readable labels
    class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

    def view_image(index=0):
        """Display a single image from the buffer with its class label."""
        plt.figure(figsize=(4, 4))
        plt.imshow(buffer_images[index])
        label_int = buffer_labels[index]
        label_name = class_names[label_int] if label_int < len(class_names) else str(label_int)
        plt.title(f"Class: {label_int} ({label_name}) | Stream Index: {index}")
        plt.axis('off')
        plt.show()

    if buffer_images:
        # IntSlider creates a draggable slider from 0 to 99
        # widgets.interactive re-renders the plot each time the slider moves
        slider = widgets.IntSlider(min=0, max=len(buffer_images)-1, value=0,
                                    description='Stream Index:')
        display(widgets.interactive(view_image, index=slider))
except Exception as e:
    print(f"Could not read from MinIO stream. {e}")